# 🎰 Sport Toto 4D — Full Data Analysis

**A rigorous, data-driven deep dive into Malaysia's Sport Toto 4D lottery.**

---
### ⚠️ Before you start
Run the scraper first to populate `data/draws.csv`:
```bash
python scraper.py --from 2019
```
Then come back here.

---
### What this notebook covers
1. Data loading & validation
2. Dataset overview & draw calendar
3. Number frequency analysis (hot/cold numbers)
4. Digit-position analysis (what digits win most in each position)
5. Gap analysis (how long numbers stay "missing")
6. Prize tier distribution over time
7. Number pair co-occurrence
8. Day-of-week patterns
9. Randomness verification (chi-square, entropy)
10. Key takeaways


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from collections import Counter
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
TOTO_RED   = '#CC0000'
TOTO_GOLD  = '#FFD700'
TOTO_DARK  = '#222222'
COOL_BLUE  = '#2196F3'

print('Libraries loaded ✅')

## 1. Load & Validate Data

In [ ]:
df = pd.read_csv('data/draws.csv', parse_dates=['date'])
df = df.sort_values('draw_seq').reset_index(drop=True)

# Columns for each prize tier
TOP3_COLS     = ['prize_1', 'prize_2', 'prize_3']
SPECIAL_COLS  = [f'special_{i}' for i in range(1, 11)]
CONSOL_COLS   = [f'consol_{i}'  for i in range(1, 11)]
ALL_NUM_COLS  = TOP3_COLS + SPECIAL_COLS + CONSOL_COLS  # 23 numbers per draw

print(f'Draws loaded      : {len(df):,}')
print(f'Date range        : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Draw seq range    : #{df["draw_seq"].min()} → #{df["draw_seq"].max()}')
print(f'Missing values    : {df[ALL_NUM_COLS].isnull().sum().sum()}')
df.head(3)

In [ ]:
# Derive useful columns
df['weekday']     = df['date'].dt.day_name()
df['week_num']    = df['date'].dt.isocalendar().week.astype(int)
df['quarter']     = df['date'].dt.quarter
df['year_month']  = df['date'].dt.to_period('M')

# Normalise all prize numbers as zero-padded strings
for col in ALL_NUM_COLS:
    df[col] = df[col].astype(str).str.zfill(4)

# Flat list of ALL drawn numbers (23 per draw)
all_numbers = df[ALL_NUM_COLS].values.flatten()
all_numbers = all_numbers[~pd.isnull(all_numbers)]
all_numbers = [n for n in all_numbers if re.match(r'^\d{4}$', str(n))]

# Flat list of TOP-3 only
top3_numbers = df[TOP3_COLS].values.flatten().tolist()

import re
print(f'Total number draws  : {len(all_numbers):,}')
print(f'Unique numbers seen : {len(set(all_numbers)):,} / 10,000 possible')
print(f'Coverage            : {len(set(all_numbers))/100:.1f}%')

## 2. Draw Calendar — Draws Per Month

In [ ]:
draws_per_month = df.groupby(['year', 'month']).size().reset_index(name='count')
pivot = draws_per_month.pivot(index='year', columns='month', values='count').fillna(0)
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]

fig, ax = plt.subplots(figsize=(14, len(pivot)*0.6 + 1))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Draws'}, ax=ax)
ax.set_title('📅 Draws Per Month (Heatmap)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig('output/01_draw_calendar.png', bbox_inches='tight')
plt.show()

print(f'Average draws/month : {draws_per_month["count"].mean():.1f}')
print(f'Draws per year      :')
print(df.groupby('year').size().to_string())

## 3. Number Frequency — Hot & Cold Numbers

In [ ]:
# Count appearances of each 4-digit number across ALL prize tiers
freq = Counter(all_numbers)
freq_df = pd.DataFrame(freq.items(), columns=['number', 'count']).sort_values('count', ascending=False)
freq_df['rank'] = range(1, len(freq_df)+1)

total_draws = len(df)
freq_df['pct_of_draws'] = freq_df['count'] / total_draws * 100

# Expected frequency if uniform
expected = len(all_numbers) / 10000
print(f'Expected appearances per number (uniform): {expected:.2f}')
print(f'Actual mean : {freq_df["count"].mean():.2f}  |  std: {freq_df["count"].std():.2f}')

print('\n🔥 TOP 20 hottest numbers (all tiers):')
print(freq_df.head(20).to_string(index=False))

print('\n🧊 TOP 20 coldest numbers (seen at least once):')
print(freq_df.tail(20).to_string(index=False))

In [ ]:
# Top/Bottom 30 bar chart
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top30 = freq_df.head(30)
axes[0].barh(top30['number'][::-1], top30['count'][::-1], color=TOTO_RED, alpha=0.85)
axes[0].axvline(expected, color='black', ls='--', lw=1.5, label=f'Expected ({expected:.1f})')
axes[0].set_title('🔥 Top 30 Most Frequent Numbers', fontweight='bold')
axes[0].set_xlabel('Times Drawn')
axes[0].legend()

bot30 = freq_df.nsmallest(30, 'count')
axes[1].barh(bot30['number'][::-1], bot30['count'][::-1], color=COOL_BLUE, alpha=0.85)
axes[1].axvline(expected, color='black', ls='--', lw=1.5, label=f'Expected ({expected:.1f})')
axes[1].set_title('🧊 Top 30 Least Frequent Numbers', fontweight='bold')
axes[1].set_xlabel('Times Drawn')
axes[1].legend()

plt.suptitle('4D Number Frequency — All Prize Tiers Combined', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('output/02_hot_cold_numbers.png', bbox_inches='tight')
plt.show()

In [ ]:
# Frequency distribution histogram — does it look uniform?
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(freq_df['count'], bins=40, color=TOTO_RED, alpha=0.7, edgecolor='white')
ax.axvline(expected, color='black', ls='--', lw=2, label=f'Expected ({expected:.1f})')
ax.axvline(freq_df['count'].mean(), color=COOL_BLUE, ls='-', lw=2, label=f'Actual mean ({freq_df["count"].mean():.1f})')
ax.set_xlabel('Times a Number Was Drawn')
ax.set_ylabel('Count of Numbers with That Frequency')
ax.set_title('Distribution of Number Frequencies\n(Should look like a bell curve centered on expected if truly random)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('output/03_frequency_distribution.png', bbox_inches='tight')
plt.show()

## 4. Digit-Position Analysis

For each of the 4 digit positions (thousands, hundreds, tens, units), which digits (0–9) appear most?

In [ ]:
# Extract digit at each position for all drawn numbers
valid_nums = [n for n in all_numbers if str(n).isdigit() and len(str(n)) == 4]

digit_pos = {}
pos_labels = ['Position 1\n(Thousands)', 'Position 2\n(Hundreds)', 'Position 3\n(Tens)', 'Position 4\n(Units)']
for pos in range(4):
    digits = [int(str(n)[pos]) for n in valid_nums]
    digit_pos[pos] = Counter(digits)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
expected_digit = len(valid_nums) / 10  # each digit equally likely

for pos, ax in enumerate(axes):
    counts = [digit_pos[pos].get(d, 0) for d in range(10)]
    bars = ax.bar(range(10), counts, color=[TOTO_RED if c == max(counts) else '#aaa' for c in counts], 
                  alpha=0.85, edgecolor='white')
    ax.axhline(expected_digit, color='black', ls='--', lw=1.5, label=f'Expected ({expected_digit:.0f})')
    ax.set_xticks(range(10))
    ax.set_xticklabels([str(d) for d in range(10)])
    ax.set_title(pos_labels[pos], fontweight='bold')
    ax.set_xlabel('Digit (0–9)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
    # Annotate max
    max_d = max(range(10), key=lambda d: digit_pos[pos].get(d, 0))
    ax.annotate(f'Most: {max_d}', xy=(max_d, max(counts)),
                xytext=(max_d+0.3, max(counts)*1.01), fontsize=9, color=TOTO_RED)

plt.suptitle('Digit Frequency by Position (All Prize Tiers)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/04_digit_position_analysis.png', bbox_inches='tight')
plt.show()

# Summary table
print('\nMost common digit per position:')
for pos in range(4):
    top = sorted(digit_pos[pos].items(), key=lambda x: -x[1])[:3]
    print(f'  Position {pos+1}: {top}')

## 5. Gap Analysis — How Long Between Appearances?

In [ ]:
# For each draw, track the last time each number appeared
# Then compute gaps (draws elapsed since last appearance)

last_seen = {}   # number → draw index
gap_records = [] # (draw_idx, number, gap)

for idx, row in df.iterrows():
    for col in ALL_NUM_COLS:
        num = str(row[col]).zfill(4)
        if not re.match(r'^\d{4}$', num):
            continue
        if num in last_seen:
            gap = idx - last_seen[num]
            gap_records.append({'draw_idx': idx, 'number': num, 'gap': gap})
        last_seen[num] = idx

gap_df = pd.DataFrame(gap_records)
print(f'Gap records: {len(gap_df):,}')
print(f'\nGap statistics (in number of draws):')
print(gap_df['gap'].describe().round(1))

# Numbers currently overdue (not seen recently)
last_draw_idx = len(df) - 1
overdue = {num: last_draw_idx - idx for num, idx in last_seen.items()}
overdue_df = pd.DataFrame(overdue.items(), columns=['number', 'draws_since_last'])
overdue_df = overdue_df.sort_values('draws_since_last', ascending=False)

print('\n🕰️  Most overdue numbers (longest gap since last appearance):')
print(overdue_df.head(20).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gap distribution
axes[0].hist(gap_df['gap'], bins=60, color=TOTO_RED, alpha=0.75, edgecolor='white')
axes[0].set_xlabel('Gap (draws between re-appearances)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Gaps Between Number Re-appearances', fontweight='bold')
median_gap = gap_df['gap'].median()
axes[0].axvline(median_gap, color='black', ls='--', lw=2, label=f'Median = {median_gap:.0f}')
axes[0].legend()

# Top 30 overdue numbers
top_overdue = overdue_df.head(30)
axes[1].barh(top_overdue['number'][::-1], top_overdue['draws_since_last'][::-1],
             color=COOL_BLUE, alpha=0.8)
axes[1].axvline(median_gap, color='black', ls='--', lw=1.5, label=f'Median gap ({median_gap:.0f})')
axes[1].set_xlabel('Draws Since Last Appearance')
axes[1].set_title('Top 30 Most Overdue Numbers', fontweight='bold')
axes[1].legend()

plt.suptitle('Gap Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/05_gap_analysis.png', bbox_inches='tight')
plt.show()

## 6. Prize Tier Distribution Over Time

In [ ]:
# Separate frequency counts by tier: 1st, 2nd, 3rd, special, consolation
tier_freqs = {}
tier_map = {
    '1st Prize': ['prize_1'],
    '2nd Prize': ['prize_2'],
    '3rd Prize': ['prize_3'],
    'Special (×10)': SPECIAL_COLS,
    'Consolation (×10)': CONSOL_COLS,
}
for tier, cols in tier_map.items():
    nums = df[cols].values.flatten().tolist()
    nums = [str(n).zfill(4) for n in nums if re.match(r'^\d{4}$', str(n))]
    tier_freqs[tier] = Counter(nums)

# Compute how many unique numbers appeared in each tier
print('Unique numbers per prize tier:')
for tier, ctr in tier_freqs.items():
    print(f'  {tier:22s}: {len(ctr):,} unique numbers, max freq = {max(ctr.values())}')

# Prize tier draw count by year
draws_by_year = df.groupby('year').size()
print('\nDraws per year:')
print(draws_by_year.to_string())

In [ ]:
# Do the same numbers tend to win at higher tiers?
# Compare top 50 numbers in 1st prize vs consolation prize
top1st = set(dict(tier_freqs['1st Prize'].most_common(100)).keys())
topConsol = set(dict(tier_freqs['Consolation (×10)'].most_common(100)).keys())
overlap = top1st & topConsol
print(f'Top-100 overlap between 1st Prize and Consolation: {len(overlap)} numbers')
print('Overlap:', sorted(overlap)[:20])

# Tier frequency comparison for top 20 numbers overall
top20_global = [n for n, _ in Counter(all_numbers).most_common(20)]
tier_compare = pd.DataFrame({
    tier: [tier_freqs[tier].get(n, 0) for n in top20_global]
    for tier in tier_map.keys()
}, index=top20_global)
print('\nTop 20 global numbers — appearances per tier:')
print(tier_compare)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
tier_compare.plot(kind='bar', ax=ax, 
                  color=[TOTO_RED, '#ff6600', '#ffaa00', COOL_BLUE, '#8bc34a'],
                  alpha=0.85, edgecolor='white', width=0.8)
ax.set_title('Top 20 Global Numbers — Appearances by Prize Tier', fontsize=14, fontweight='bold')
ax.set_xlabel('Number')
ax.set_ylabel('Times Drawn')
ax.legend(title='Prize Tier', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('output/06_tier_distribution.png', bbox_inches='tight')
plt.show()

## 7. Number Pair Co-occurrence (1st Prize Focus)

Do any two numbers tend to appear in the same draw (across all tiers)?

In [ ]:
# For each draw, look at all 23 numbers and count pairs
# Focus on top 50 most frequent numbers to keep it tractable
top50 = [n for n, _ in Counter(all_numbers).most_common(50)]

pair_counts = Counter()
for idx, row in df.iterrows():
    draw_nums = set()
    for col in ALL_NUM_COLS:
        n = str(row[col]).zfill(4)
        if n in top50:
            draw_nums.add(n)
    for a, b in combinations(sorted(draw_nums), 2):
        pair_counts[(a, b)] += 1

print('Top 20 most co-occurring pairs (among top-50 numbers):')
for pair, cnt in pair_counts.most_common(20):
    print(f'  {pair[0]} + {pair[1]} : {cnt} times')

In [ ]:
# Heatmap of co-occurrence for top 25 numbers
top25 = [n for n, _ in Counter(all_numbers).most_common(25)]
matrix = pd.DataFrame(0, index=top25, columns=top25)

for idx, row in df.iterrows():
    draw_nums = []
    for col in ALL_NUM_COLS:
        n = str(row[col]).zfill(4)
        if n in top25:
            draw_nums.append(n)
    for a, b in combinations(draw_nums, 2):
        if a in matrix.index and b in matrix.columns:
            matrix.loc[a, b] += 1
            matrix.loc[b, a] += 1

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.eye(len(top25), dtype=bool)
sns.heatmap(matrix, mask=mask, cmap='YlOrRd', annot=False,
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Co-occurrence count'})
ax.set_title('Number Co-occurrence Heatmap (Top 25 Numbers)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/07_cooccurrence_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Day-of-Week Patterns

Does any weekday produce different winning numbers?

In [ ]:
weekday_order = ['Wednesday', 'Saturday', 'Sunday']
weekday_counts = df['weekday'].value_counts()
print('Draws by weekday:')
print(weekday_counts.to_string())

# For each weekday, get the top-10 most frequent 1st prize numbers
for day in weekday_order:
    sub = df[df['weekday'] == day]
    top = Counter(sub['prize_1'].tolist()).most_common(10)
    print(f'\n{day} — Top 10 1st Prize numbers:')
    print(top)

In [ ]:
# Compare digit distribution across weekdays
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, day in zip(axes, weekday_order):
    sub = df[df['weekday'] == day]
    nums_day = sub[ALL_NUM_COLS].values.flatten()
    nums_day = [n for n in nums_day if re.match(r'^\d{4}$', str(n))]
    digit_counts = Counter(int(d) for n in nums_day for d in str(n).zfill(4))
    vals = [digit_counts.get(d, 0) for d in range(10)]
    expected_d = sum(vals) / 10
    bars = ax.bar(range(10), vals, color=TOTO_RED, alpha=0.75, edgecolor='white')
    ax.axhline(expected_d, color='black', ls='--', lw=1.5, label=f'Expected')
    ax.set_title(f'{day}\n({len(sub)} draws)', fontweight='bold')
    ax.set_xticks(range(10))
    ax.set_xticklabels([str(d) for d in range(10)])
    ax.set_xlabel('Digit')
    if ax == axes[0]:
        ax.set_ylabel('Digit Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Overall Digit Frequency by Draw Day\n(All positions combined)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/08_weekday_patterns.png', bbox_inches='tight')
plt.show()

## 9. Randomness Verification — Chi-Square & Entropy

In [ ]:
from scipy.stats import chisquare, entropy as scipy_entropy

# --- Chi-square test: is the number distribution uniform? ---
# Only use numbers in 0000–9999 range
all_valid = [n for n in all_numbers if re.match(r'^\d{4}$', str(n))]
observed = Counter(all_valid)

# Fill in 0 for unseen numbers
all_possible = [f'{i:04d}' for i in range(10000)]
observed_counts = np.array([observed.get(n, 0) for n in all_possible])
expected_counts = np.full(10000, len(all_valid) / 10000)

chi2_stat, p_value = chisquare(observed_counts, expected_counts)

print('=== Chi-Square Test (Uniformity of Number Distribution) ===')
print(f'Chi² statistic : {chi2_stat:,.2f}')
print(f'p-value        : {p_value:.4f}')
print(f'Result         : {"Cannot reject uniformity (looks random!)" if p_value > 0.05 else "Deviation from uniformity detected"}')

# --- Per-digit chi-square for each position ---
print('\n=== Per-Digit Chi-Square by Position ===')
for pos in range(4):
    digits = [int(str(n)[pos]) for n in all_valid]
    obs = [digits.count(d) for d in range(10)]
    exp = [len(digits) / 10] * 10
    c2, p = chisquare(obs, exp)
    print(f'  Position {pos+1}: chi²={c2:.2f}, p={p:.4f} → {"✅ Uniform" if p > 0.05 else "⚠️ Skewed"}')

# --- Shannon Entropy ---
probs = observed_counts / observed_counts.sum()
probs = probs[probs > 0]
entropy_bits = scipy_entropy(probs, base=2)
max_entropy = np.log2(10000)  # perfect uniform = ~13.29 bits
print(f'\n=== Shannon Entropy ===')
print(f'Observed entropy : {entropy_bits:.4f} bits')
print(f'Max possible     : {max_entropy:.4f} bits (perfect uniform)')
print(f'Entropy ratio    : {entropy_bits/max_entropy*100:.2f}%')
print(f'Interpretation   : {"Very close to random" if entropy_bits/max_entropy > 0.98 else "Some structure present"}')

In [ ]:
# Visual: Q-Q plot comparing observed frequency distribution to expected uniform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Observed vs expected frequency of each number
axes[0].scatter(expected_counts[:500], observed_counts[:500], alpha=0.3, s=8, color=TOTO_RED)
lim = max(observed_counts.max(), expected_counts.max()) * 1.1
axes[0].plot([0, lim], [0, lim], 'k--', lw=1.5, label='Perfect uniform')
axes[0].set_xlabel('Expected count')
axes[0].set_ylabel('Observed count')
axes[0].set_title('Expected vs Observed Frequency\n(first 500 numbers shown)', fontweight='bold')
axes[0].legend()

# Rolling entropy over time — does randomness change over years?
window = 52  # ~1 year of draws
rolling_entropy = []
for i in range(window, len(df)):
    subset = df.iloc[i-window:i]
    nums = subset[ALL_NUM_COLS].values.flatten()
    nums = [n for n in nums if re.match(r'^\d{4}$', str(n))]
    cnt = Counter(nums)
    p = np.array(list(cnt.values())) / len(nums)
    ent = scipy_entropy(p, base=2)
    rolling_entropy.append((df.iloc[i]['date'], ent))

re_df = pd.DataFrame(rolling_entropy, columns=['date', 'entropy'])
axes[1].plot(re_df['date'], re_df['entropy'], color=TOTO_RED, lw=1.5)
axes[1].axhline(re_df['entropy'].mean(), color='black', ls='--', lw=1, label=f'Mean: {re_df["entropy"].mean():.2f}')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Shannon Entropy (bits)')
axes[1].set_title(f'Rolling Entropy ({window}-draw window)\nHigher = more random', fontweight='bold')
axes[1].legend()

plt.suptitle('Randomness Verification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/09_randomness_verification.png', bbox_inches='tight')
plt.show()

## 10. 1st Prize Number Streak & Recurrence Analysis

In [ ]:
# Has any 1st prize number ever repeated?
p1_series = df['prize_1'].tolist()
p1_counter = Counter(p1_series)
repeated_p1 = {k: v for k, v in p1_counter.items() if v > 1}
print(f'Total unique 1st prize numbers: {len(p1_counter)}')
print(f'Numbers that won 1st prize more than once: {len(repeated_p1)}')
if repeated_p1:
    print('\nRepeated 1st prize numbers:')
    for num, cnt in sorted(repeated_p1.items(), key=lambda x: -x[1]):
        draws_won = df[df['prize_1'] == num][['draw_seq','date']].to_dict('records')
        print(f'  {num}: won {cnt}× — draws: {draws_won}')

# How quickly does a 1st prize number re-appear in ANY position?
p1_reappear = []
for i, row in df.iterrows():
    p1 = row['prize_1']
    for j in range(i+1, min(i+200, len(df))):
        future = df.iloc[j]
        if p1 in [str(future[c]).zfill(4) for c in ALL_NUM_COLS]:
            p1_reappear.append(j - i)
            break

print(f'\n1st prize number re-appearance in any position:')
print(f'  Median draws until re-appearance : {np.median(p1_reappear):.0f}')
print(f'  Mean   draws until re-appearance : {np.mean(p1_reappear):.1f}')
print(f'  Min    draws until re-appearance : {np.min(p1_reappear)}')
print(f'  Max    draws until re-appearance : {np.max(p1_reappear)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1st prize frequency distribution
top1_vals = list(p1_counter.values())
axes[0].hist(top1_vals, bins=range(1, max(top1_vals)+2), color=TOTO_RED, alpha=0.75, edgecolor='white', align='left')
axes[0].set_xlabel('Times a Number Won 1st Prize')
axes[0].set_ylabel('Count of Numbers')
axes[0].set_title('How Many Times Has Each 1st Prize Number Won?', fontweight='bold')
axes[0].set_xticks(range(1, max(top1_vals)+1))

# Re-appearance distribution
axes[1].hist(p1_reappear, bins=30, color=COOL_BLUE, alpha=0.75, edgecolor='white')
axes[1].axvline(np.median(p1_reappear), color='black', ls='--', lw=2, label=f'Median={np.median(p1_reappear):.0f} draws')
axes[1].set_xlabel('Draws Until 1st Prize Number Re-appears Anywhere')
axes[1].set_ylabel('Frequency')
axes[1].set_title('How Quickly Does a 1st Prize Number Come Back?', fontweight='bold')
axes[1].legend()

plt.suptitle('1st Prize Recurrence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/10_first_prize_recurrence.png', bbox_inches='tight')
plt.show()

## 11. Full Number Heatmap (All 10,000 Numbers)

In [ ]:
# Create 100×100 heatmap: row = first 2 digits (00–99), col = last 2 digits (00–99)
heatmap_data = np.zeros((100, 100))

for num, cnt in observed.items():
    row = int(str(num).zfill(4)[:2])
    col = int(str(num).zfill(4)[2:])
    heatmap_data[row, col] = cnt

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Times Drawn')
ax.set_xlabel('Last 2 digits (00–99)')
ax.set_ylabel('First 2 digits (00–99)')
ax.set_title('Full 4D Number Space Heatmap\n(Row = first 2 digits, Col = last 2 digits)', 
             fontsize=14, fontweight='bold')

# Add tick labels every 10
ticks = list(range(0, 100, 10))
ax.set_xticks(ticks)
ax.set_xticklabels([f'{t:02d}' for t in ticks])
ax.set_yticks(ticks)
ax.set_yticklabels([f'{t:02d}' for t in ticks])

plt.tight_layout()
plt.savefig('output/11_full_number_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Numbers never drawn: {(heatmap_data == 0).sum():,}')
print(f'Numbers drawn 1× : {(heatmap_data == 1).sum():,}')
print(f'Numbers drawn 5+× : {(heatmap_data >= 5).sum():,}')

## 12. Key Takeaways & Summary

In [ ]:
# ── Final summary printout ──────────────────────────────────────────────────
hottest = freq_df.head(5)['number'].tolist()
coldest = freq_df.tail(5)['number'].tolist()
coverage = len(set(all_numbers)) / 100

print('=' * 60)
print('        SPORT TOTO 4D — ANALYSIS SUMMARY')
print('=' * 60)
print(f'  Dataset        : {len(df):,} draws | {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Total numbers  : {len(all_valid):,} drawn')
print(f'  Coverage       : {coverage:.1f}% of all possible 4D numbers have appeared')
print()
print(f'  🔥 Hottest numbers  : {", ".join(hottest)}')
print(f'  🧊 Coldest numbers  : {", ".join(coldest)}')
print()
print(f'  Chi-square p-value : {p_value:.4f}')
print(f'  Entropy ratio      : {entropy_bits/max_entropy*100:.2f}%')
print()
print('  VERDICT:')
if p_value > 0.05 and entropy_bits/max_entropy > 0.97:
    print('  ✅ The draw is statistically consistent with a truly')
    print('     random process. No systematic bias detected.')
    print('     Hot/cold patterns are statistical noise, not signal.')
else:
    print('  ⚠️  Some deviation from perfect uniformity was detected.')
    print('     Investigate further before drawing conclusions.')
print()
print('  The bottom line: play for fun, not profit. 🎰')
print('=' * 60)

---
## Appendix: Export Summary Tables to CSV

In [ ]:
os.makedirs('data', exist_ok=True)

# 1. Number frequency table
freq_df.to_csv('data/number_frequency.csv', index=False)
print('Saved: data/number_frequency.csv')

# 2. Overdue numbers
overdue_df.to_csv('data/overdue_numbers.csv', index=False)
print('Saved: data/overdue_numbers.csv')

# 3. Gap stats
gap_df.to_csv('data/gap_analysis.csv', index=False)
print('Saved: data/gap_analysis.csv')

# 4. Top pairs
pairs_list = [{'num_a': a, 'num_b': b, 'cooccurrences': cnt}
              for (a,b), cnt in pair_counts.most_common(200)]
pd.DataFrame(pairs_list).to_csv('data/number_pairs.csv', index=False)
print('Saved: data/number_pairs.csv')

print('\n✅ All exports complete!')